## 专家知识工作者

### 一个问答智能体，扮演专家知识工作者
### 供保险科技公司 Insurellm 的员工使用
### 智能体需要准确，且方案应低成本。

本项目将使用 RAG（Retrieval Augmented Generation，检索增强生成），确保我们的问答助手具有高准确率。

## 今日内容：

- Part A：我们将文档切分为 CHUNKS（块）
- Part B：我们将 CHUNKS 编码为 VECTORS（向量）并放入 Chroma
- Part C：我们将可视化这些向量

### PART A：将文档切分为块

In [ ]:
# 导入：tiktoken 数 token；LangChain 的 Embedding / Chroma / 文档加载与切分；
# HuggingFaceEmbeddings 本地嵌入模型；TSNE + plotly 可视化向量

import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [ ]:
# 价格对我们公司很重要，所以我们使用低成本模型
# db_name：向量数据库（vector DB / Chroma）的本地持久化目录

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


In [ ]:
# 所有文档一共有多少字符？
# 先用 glob 递归读取 knowledge-base 下全部 Markdown，拼成一大段文本

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

In [ ]:
# 所有文档一共有多少 token？
# token：模型计费与上下文窗口的单位；tiktoken 按模型规则做分词（tokenize）

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

In [ ]:
# 使用 LangChain 的 DirectoryLoader / TextLoader 加载知识库中的全部内容
# 并为每个 Document 写入 metadata["doc_type"]（产品/员工等），供后续过滤与可视化

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

In [ ]:
# 查看第 2 个 Document 对象（含 page_content 与 metadata）

documents[1]

In [ ]:
# 使用 RecursiveCharacterTextSplitter 切分为 chunk（文本块）
# chunk_size=1000：块大小；chunk_overlap=200：重叠，避免关键信息落在切缝上

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

In [ ]:
# 随便看一个 chunk（文本块），体会切分后的粒度

chunks[100]

### PART B：生成向量并存储到 Chroma

在第 3 周，你已设置 Hugging Face 账户并获得了 HF_TOKEN

此时，你可能想把它加入 `.env` 文件，并运行 `load_dotenv(override=True)`

（实际上这不应该是必需的。）

In [ ]:
# 选择一个嵌入模型（embedding model）：把文本变成向量
# HuggingFace 的 all-MiniLM-L6-v2 可本地运行；也可用 OpenAI 嵌入（下一行注释掉了）
# Chroma：向量数据库（vector DB），负责存储与相似度检索

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

In [ ]:
# 让我们检查一下这些向量

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

### Part C：可视化！

In [ ]:
# 准备工作：取出全部向量、原文与元数据，并按文档类型上色（便于可视化）

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
# 我们人类更容易在二维中可视化事物！
# 使用 t-SNE 将向量降维到 2D
# （t-distributed stochastic neighbor embedding，t 分布随机邻域嵌入）

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# 创建 2D 散点图
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# 试试 3D！

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# 创建 3D 散点图
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()